In [1]:
import numpy as np
from scipy import constants
import pandas as pd
pd.set_option('display.width', 10000) # Adjust for desired width
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None) # display full content in a column
import matplotlib.pyplot as plt
from tweezer_functions import * 
from IonChainTools import *
from scipy.optimize import fsolve
import matplotlib.colors as mcolors
import matplotlib.colorbar as mcolorbar
from scipy.optimize import fsolve
from scipy.optimize import curve_fit
from scipy.optimize import minimize
import matplotlib.ticker as ticker
import itertools
import math

#Constants in SI units
eps0 = constants.epsilon_0 
m = 39.9626*constants.atomic_mass
c = constants.c
e = constants.e
hbar = constants.hbar
pi = np.pi

# setting up parameters that we're not changing
qubit_wavelength = 729e-9
tweezer_wavelength = 532e-9
omega_tweezer = 2*pi*c/tweezer_wavelength
print(omega_tweezer)
df = pd.read_csv("S_P_only.csv",sep = ",",encoding = "UTF-8")
lambdares = np.array(df["wavelength (nm)"])*1e-9
omega_res = 2*pi*c/lambdares
linewidths = np.array(df["A_ki (s^-1)"])
lifetimes = linewidths
print(linewidths)
#test

3540698434791077.0
[1.47e+08 1.40e+08]


# testing functions

In [2]:
def equally_spaced_positions(N,d):
    if N % 2 == 1:
        positions = np.arange(-(N//2), N//2 + 1) * d
    else:
        # for even N, center between two ions
        positions = (np.arange(-N/2 + 0.5, N/2 + 0.5)) * d
    positions_list = positions.tolist()
    return positions_list

In [3]:
def ion_force_quartic(positions, N, k4, omega_rf_axial):
    """
    Calculate force on each ion.
    
    Inputs:
    -----------
    positions : array-like
        Position of each ion (length N)
    N : int
        Number of ions
    k4 : float
        Quartic interaction coefficient
    omega_rf_axial : float
        Axial trapping frequency from rf potential (2*pi*Hz)
    
    Returns:
    --------
    force_list : list
        Force on each ion (length N)
    """
    x = np.array(positions)
    N = len(x)
    A = 0.5 * m * omega_rf_axial**2
    B = (e**2) / (4 * pi * eps0)
    C = k4

    return [2*A*(x[m]) + 4*C * (x[m]**3) 
            - sum([B * (x[m] - x[n]) / (abs(x[m] - x[n])**3) for n in range(m) if x[m] != x[n]])  # Avoid division by zero
            + sum([B * (x[m] - x[n]) / (abs(x[m] - x[n])**3) for n in range(m+1, N) if x[m] != x[n]])  # Avoid division by zero
           
            for m in range(N)]


N = 15
k4 = 0.00177
omega_rf_axial = 150e3*2*pi
estimated_extreme = 0.481*N**0.765
x0 = np.linspace(-estimated_extreme, estimated_extreme, N)

ueq_test = fsolve(ion_force_quartic, x0, args=(N, k4, omega_rf_axial))
diff_list = []
for x, y in zip(ueq_test[0::], ueq_test[1::]):
    diff_list.append(y-x)
diff_list

/Users/ritika/anaconda3/lib/python3.10/site-packages/scipy/optimize/_minpack_py.py:178: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  warnings.warn(msg, RuntimeWarning)


[5.231334134260145e-06,
 4.323839194694244e-06,
 4.72025407683626e-06,
 3.44286201413891e-06,
 7.1912297104941415e-06,
 4.469411295911118e-06,
 -7.2870498116343415e-06,
 1.781096366938332e-05,
 3.011792890101129e-06,
 -7.898670131382756e-06,
 9.874248939254035e-06,
 2.2413323608826758e-06,
 3.211509994864699e-06,
 1.7354252790722454e-06]

# trying functions

### I am assuming the potential is quartic in the axial direction and magically creates equally spaced ions.  In the radial direction, it is magically quadratic and my radial Hessian is analytically unchanged, only the equilibrium positions are changed. 

Make a function that generates equally spaced ions for whatever N I want, and then use that function in all of my other functions.  see what else in the pipeline needs to change to accomodate this change.

1. given some distance d, make a list that puts the center ion at z=0 and spaces the remaining N ions equally spaced from the center ion
2. go through the pipeline to put in equlibrium positions as a function input
3. run functions

In [4]:
N = [3,4]
w_rf_a = 0.125e6*2*np.pi
w_rf_r = 4e6*2*np.pi
f_rf_r = w_rf_r/(2*pi)
f_rf_a = w_rf_a/(2*pi)

tweezed_ion = [0]
P_opt = [100e-3]
w0 = 1e-6
U = potential(omega_tweezer,linewidths,omega_res,P_opt[0],w0)

In [5]:
ueq_dict = {N:ion_spacing(N,w_rf_a)[0] for N in N}
print(ueq_dict[3])

[-1.91703424e-05  3.21474771e-19  1.91703424e-05]


In [6]:
many_N = []

for N in range(3, 5):
    test_loop = midcircuit_modes_untweezed(
        omega_tweezer,
        linewidths,
        omega_res,
        m,
        mode_calc_r,
        N,
        f_rf_r,
        ueq_dict,
        P_opt,
        w0,
    )
    many_N.append((N, test_loop))

In [7]:
many_N

[(3,
     Tweezed Ion  Coolant Ion                                                    Mode couplings                                       inverse_mode_couplings  sum_inverse_middle
  0          NaN            0  [0.027979725457398488, -0.0342763965840775, 0.01979626318834981]     [35.7401648390934, 29.17459533842984, 50.51458401444695]          115.429344
  1          NaN            2   [0.027979725457406047, 0.03427639658407677, 0.0197962631883501]  [35.74016483908375, 29.174595338430468, 50.514584014446214]          115.429344),
 (4,
     Tweezed Ion  Coolant Ion                                                                            Mode couplings                                                          inverse_mode_couplings  sum_inverse_middle
  0          NaN            0   [0.02423115303703591, -0.032681114076025176, -0.02424540129267283, 0.01034317157808717]     [41.2691875814394, 30.59871207798264, 41.24493498493707, 96.68214362010386]          209.794978
  1          NaN 